In [1]:
# text data로 RAG 구현
# data(file) loading -> embedding -> vectorDB에 저장 -> vectorDB에서 검색(R)
# -> prompt 내용 강화(A) -> LLM에 질문(G) -> 결과 얻기

!pip install -U langchain langchain-core langchain-community chromadb langchain-google-genai google-genai langchain-openai python-dotenv
!pip install -U sentence-transformers
!pip install -U langchain-text-splitters

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 8.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.0/475.0 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.0/262.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 68.3 MB/s eta 0

In [11]:
import os, io
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=api_key)
llm = genai.GenerativeModel("gemini-2.5-flash")

embedder = SentenceTransformer('all-MiniLM-L6-v2')

# text 읽기 1 - raw
# documents = [
#     "김치찌개는 한국의 대표적인 찌개 요리이다.",
#     "된장찌개는 발효된 된장을 이용해 만든다.",
#     "비빔밥은 여러 가지 나물을 비벼서 먹는 밥 요리이다.",
#     "불고기는 양념한 소고기를 구워 먹는 전통 음식이다.",
#     "삼계탕은 닭에 인삼, 대추, 생강, 찹쌀 등을 넣고 푹 끓인 보양식이다."
# ]

# text 읽기 2 - python 방식
# with open("foods.txt", "r", encoding="utf-8") as f:
#     documents = [line.strip() for line in f if line.strip()]

# text 읽기 3 - langchain 방식
# from langchain_community.document_loaders import TextLoader
# loader = TextLoader("foods.txt", encoding="utf-8")
# datas = loader.load()
# print(datas)    # List type : [Document(metadata={'source': ...
# documents = [doc.page_content for doc in datas]
# documents = datas[0].page_content.split("\n")
# documents = [doc.strip() for doc in documents if doc.strip()]

# text 읽기 4 - Langchain 방식 (줄마다 분리)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter # Corrected import path
loader = TextLoader("foods.txt", encoding="utf-8")
datas = loader.load()
# splitter = CharacterTextSplitter(separator="\n", chunk_size=100, chunk_overlap=0)
# spl_docs = splitter.split_documents(datas)
# documents = [doc.page_content for doc in spl_docs] # Corrected to use spl_docs
# print(len(documents))
text = datas[0].page_content
lines = text.split("\n")
# max_len = max([len(line.strip()) for line in lines])
max_len = max(len(line) for line in lines if line.strip())
splitter = CharacterTextSplitter(separator="\n", chunk_size=max_len, chunk_overlap=0)
chunks = splitter.split_text(text)
documents = [c.strip() for c in chunks if c.strip()]


print(documents)

['김치찌개는 한국의 대표적인 찌개 요리이다.', '된장찌개는 발효된 된장을 이용해 만든다.', '비빔밥은 여러 가지 나물을 비벼서 먹는 밥 요리이다.', '불고기는 양념한 소고기를 구워 먹는 전통 음식이다.', '삼계탕은 닭에 인삼, 대추, 생강, 찹쌀 등을 넣고 푹 끓인 보양식이다.']


In [13]:
# 임베딩 벡터로 변환
doc_embeddings = embedder.encode(documents)
print(doc_embeddings[0][:5])

# ChromaDB에 저장 (Faiss도 있음)
# 방법 1. 임시 저장
# chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# 방법 2. 영구 저장
chroma_client = chromadb.Client(Settings(
    persist_directory="./chroma_db",
    anonymized_telemetry=False            # 보안 강화 목적
))

collection = chroma_client.get_or_create_collection(name="foods")

for i, (doc, embedding) in enumerate(zip(documents, doc_embeddings)):
    collection.add(
        documents=[doc],              # 담을 때 List type으로 -> chromaDB는 list를 원함
        embeddings=[embedding.tolist()],
        ids=[f"doc_{i}"]
    )

[-0.00259106  0.06818165  0.03677396  0.01112984 -0.05619021]


In [21]:
# ------------------- 여기서부터 RAG 단계 따라 처리 ------------------------
# RAG 흐름 1단계 : Retrival 관련 문서 검색
query = "한국의 대표적인 찌개 음식이랑 그 레시피 알려줘."
query_embedding = embedder.encode(query)

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    # n_results=3
    n_results=len(documents),
    include=["documents", "distances"]
)

print(results)

# 유사도 거리 확인
import numpy as np
cosine_distances = results["distances"][0]
print(f"코사인 거리 : {cosine_distances}")

similarities = []
# 유사도 직접 계산
for doc_id in results["ids"][0]:
  doc_embed = collection.get(ids=[doc_id], include=["embeddings"])["embeddings"][0]
  doc_embed = np.array(doc_embed, dtype=float)     # 내적 연산하기 위해서 정규화 필요
  # np.dot 사용 여부 확인 : 정규화 확인 -> 1.0 에 근사하면 ok
  # print(f"정규화 확인 1 : {np.linalg.norm(query_embedding)}")
  # print(f"정규화 확인 2 : {np.linalg.norm(doc_embed)}")

  sim = np.dot(query_embedding, doc_embed)
  similarities.append(sim)

print(f"유사도 값 확인 : {similarities}")

{'ids': [['doc_2', 'doc_3', 'doc_0', 'doc_4', 'doc_1']], 'embeddings': None, 'documents': [['비빔밥은 여러 가지 나물을 비벼서 먹는 밥 요리이다.', '불고기는 양념한 소고기를 구워 먹는 전통 음식이다.', '김치찌개는 한국의 대표적인 찌개 요리이다.', '삼계탕은 닭에 인삼, 대추, 생강, 찹쌀 등을 넣고 푹 끓인 보양식이다.', '된장찌개는 발효된 된장을 이용해 만든다.']], 'uris': None, 'included': ['documents', 'distances'], 'data': None, 'metadatas': None, 'distances': [[0.5128312110900879, 0.5347613096237183, 0.5555259585380554, 0.8822628855705261, 1.0535786151885986]]}
코사인 거리 : [0.5128312110900879, 0.5347613096237183, 0.5555259585380554, 0.8822628855705261, 1.0535786151885986]
유사도 값 확인 : [np.float64(0.7435844283018229), np.float64(0.7326192914005547), np.float64(0.7222369603004755), np.float64(0.5588685103065907), np.float64(0.47321063359170756)]


In [25]:
# RAG 흐름 2, 3 단계 : 증강(Augumented) 및 생성(Generation)
# 똑똑한 프롬프트 만들기
# 검색된 문자열 하나로 합치기
retrieved_docs_list = results["documents"][0]
retrieved_docs = "\n".join(retrieved_docs_list)
# print(retrieved_docs)

prompt = f"""
  너는 한국 전통 음식에 대해 잘 아는 전문가야.
  지금부터 사용자 질문에 답변할 때는 반드시 아래 내용을 참고해서 답변해줘.
  {retrieved_docs}

  위 내용을 참고하여 '{query}'에 대해 대답해줘.
  10 문장 이내로, 마크다운(**, *, -, bullet point ...) 같은 스타일 없이 평문으로 답해줘.
"""

response = llm.generate_content(prompt)

print("LLM 응답 :")
# print(response.text)

if hasattr(response, "text"):
  text = response.text
else:
  text = response.candidates[0].content.parts[0].text

# 문장 분리
import re
sentences = re.split(r'\.\s+', text)
for s in sentences:
  s = s.strip()
  if not s:
    continue
  if not s.endswith("."):
    s += "."
  print(s)

LLM 응답 :
한국의 대표적인 찌개 요리는 바로 김치찌개입니다.
이 요리는 한국의 식탁에서 빼놓을 수 없는 중요한 음식 중 하나입니다.
김치찌개는 잘 익은 김치를 주재료로 하여 돼지고기나 참치, 두부, 파, 양파 등을 넣고 얼큰하게 끓여냅니다.
레시피는 간단합니다.
먼저 돼지고기나 참치를 잘게 썰어 볶다가 김치를 넣고 함께 볶아줍니다.
그 후 육수나 물을 붓고 두부, 파, 양파 등 기호에 맞는 재료를 더하여 충분히 끓여주면 됩니다.
특히 국물이 보글보글 끓어오르며 재료들의 맛이 어우러질 때 깊은 맛이 납니다.
얼큰하고 시원한 맛이 특징이며, 밥과 함께 먹으면 속이 든든해지는 한국인의 소울 푸드라 할 수 있습니다.
또한 발효된 된장을 이용해 만드는 된장찌개도 한국인이 즐겨 찾는 대표적인 찌개 요리 중 하나입니다.
